In [1]:
import pandas as pd
import ast
import pickle
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# =====================================================
# STEP 1: Load datasets
# =====================================================
movies = pd.read_csv("data/tmdb_5000_movies.csv")
credits = pd.read_csv("data/tmdb_5000_credits.csv")

print("Datasets loaded")

# =====================================================
# STEP 2: Merge datasets
# =====================================================
movies = movies.merge(credits, on="title")
print("Datasets merged")

# =====================================================
# STEP 3: Select useful columns
# =====================================================
movies = movies[
    [
        "movie_id",
        "title",
        "overview",
        "genres",
        "keywords",
        "cast",
        "crew"
    ]
]

movies.dropna(inplace=True)
print("Columns filtered")

# =====================================================
# STEP 4: Helper functions to parse JSON columns
# =====================================================
def extract_names(obj):
    names = []
    for i in ast.literal_eval(obj):
        names.append(i["name"])
    return names

def extract_top_cast(obj, top_n=7):
    cast_list = ast.literal_eval(obj)
    return " ".join([i["name"] for i in cast_list][:top_n])

def extract_director(obj):
    crew_list = ast.literal_eval(obj)
    for i in crew_list:
        if i["job"] == "Director":
            return i["name"]
    return ""

# =====================================================
# STEP 5: Apply feature extraction
# =====================================================
movies["genres"] = movies["genres"].apply(extract_names).apply(lambda x: " ".join(x))
movies["keywords"] = movies["keywords"].apply(extract_names).apply(lambda x: " ".join(x))
movies["cast_names"] = movies["cast"].apply(extract_top_cast)
movies["director_name"] = movies["crew"].apply(extract_director)

print("Metadata extracted")

# =====================================================
# STEP 6: Create tags column for vectorization
# =====================================================
movies["tags"] = (
    movies["overview"] + " " +
    movies["genres"] + " " +
    movies["keywords"] + " " +
    movies["cast_names"] + " " +
    movies["director_name"]
)

movies["tags"] = movies["tags"].str.lower()

print("Tags column built")

# =====================================================
# STEP 7: Vectorization
# =====================================================
cv = CountVectorizer(
    max_features=7000,
    stop_words="english",
    ngram_range=(1, 2)
)

vectors = cv.fit_transform(movies["tags"]).toarray()

print("Vectorization complete")

# =====================================================
# STEP 8: Similarity matrix
# =====================================================
similarity = cosine_similarity(vectors)

print("Similarity matrix computed")

# =====================================================
# STEP 9: Save processed data
# =====================================================
pickle.dump(movies, open("models/movies.pkl", "wb"))
pickle.dump(vectors, open("models/vectors.pkl", "wb"))
pickle.dump(cv, open("models/vectorizer.pkl", "wb"))
pickle.dump(similarity, open("models/embeddings.pkl", "wb"))

print("Model files saved successfully")

Datasets loaded
Datasets merged
Columns filtered
Metadata extracted
Tags column built
Vectorization complete
Similarity matrix computed
Model files saved successfully


In [2]:
import pickle
movies = pickle.load(open("models/movies.pkl", "rb"))
print(movies.columns)


Index(['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew',
       'cast_names', 'director_name', 'tags'],
      dtype='object')
